## LLM-assisted annotation step

I added an LLM-assisted annotation step to classify value themes and identify linguistic shifts in the archived About / Mission / Values pages. The prompt compares each company-year page with the prior-year page when available and returns JSON fields for theme categories, changed-from-prior status, linguistic-shift notes, and analyst notes.

In this submitted version, I keep the LLM step as a reproducible pipeline design rather than a completed full annotation layer, because the API call could not be completed due to quota limits. The full structured dataset therefore relies on the keyword-based coding used above.

In [1]:
import sys
!{sys.executable} -m pip install --upgrade openai typing_extensions

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [3]:
import os
import json
import pandas as pd
from openai import OpenAI

In [4]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
def build_llm_prompt(row, prior_text):
    current_text = row.get("page_text_clean", "")

    current_text = current_text[:4000] if isinstance(current_text, str) else ""
    prior_text = prior_text[:3000] if isinstance(prior_text, str) else ""

    prompt = f"""
You are helping analyze archived corporate About / Mission / Values pages.

Company: {row['company_name']}
Ticker: {row['ticker']}
Sector: {row['sector']}
Year: {row['year']}

Prior-year page text:
{prior_text if prior_text else "No prior-year text available."}

Current-year page text:
{current_text}

Please analyze only the text provided. Do not use outside knowledge.

Return a valid JSON object with exactly these fields:
- llm_changed_from_prior: true, false, or null if prior-year text is unavailable
- llm_theme_categories: a list of 3-6 value themes present in the current page
- llm_linguistic_shift_notes: 1-2 sentences about any notable change in language, emphasis, or framing
- llm_analyst_notes: 1-2 sentences summarizing the main values emphasized by the page

Keep the language concise and analytical.
"""
    return prompt

In [6]:
def analyze_with_llm(row, prior_text):
    prompt = build_llm_prompt(row, prior_text)

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "You are a careful research assistant analyzing corporate values language."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            response_format={"type": "json_object"},
            temperature=0
        )

        result = response.choices[0].message.content
        return json.loads(result)

    except Exception as e:
        return {
            "llm_changed_from_prior": None,
            "llm_theme_categories": [],
            "llm_linguistic_shift_notes": "",
            "llm_analyst_notes": f"LLM analysis failed: {e}"
        }

In [7]:
part1 = pd.read_csv("../data/part1_company_year_values.csv")
usable = part1[
    part1["page_text_clean"].notna() &
    (part1["page_text_clean"].astype(str).str.len() > 200)
].copy()

llm_sample = (
    usable
    .sort_values(["sector", "ticker", "year"])
    .groupby("sector")
    .head(10)
    .copy()
)

llm_sample[["ticker", "company_name", "sector", "year"]].head()

,ticker,company_name,sector,year
28,AMZN,Amazon,Consumer Discretionary,2017
29,AMZN,Amazon,Consumer Discretionary,2018
30,AMZN,Amazon,Consumer Discretionary,2019
33,AMZN,Amazon,Consumer Discretionary,2022
34,AMZN,Amazon,Consumer Discretionary,2023


In [8]:
def get_prior_text(df, ticker, year):
    prior = df[
        (df["ticker"] == ticker) &
        (df["year"] == year - 1)
    ]

    if prior.empty:
        return ""

    text = prior.iloc[0].get("page_text_clean", "")
    return text if isinstance(text, str) else ""

In [9]:
test_row = llm_sample.iloc[0]
test_prior = get_prior_text(part1, test_row["ticker"], test_row["year"])

test_result = analyze_with_llm(test_row, test_prior)
test_result

{'llm_changed_from_prior': None,
 'llm_theme_categories': [],
 'llm_linguistic_shift_notes': '',
 'llm_analyst_notes': "LLM analysis failed: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}"}